In [0]:
print("conectado")

In [0]:
%pip install boto3

In [0]:
access_key = dbutils.secrets.get(scope="aws", key="access_key_id")
secret_key = dbutils.secrets.get(scope="aws", key="secret_access_key")

import boto3
s3 = boto3.client(
    "s3",
    aws_access_key_id=access_key,
    aws_secret_access_key=secret_key,
    region_name="us-east-2",
)

response = s3.list_objects_v2(Bucket="eduardo-personal-data-projects", Prefix="raw/nyc_taxi/")
for obj in response.get("Contents", []):
    print(obj["Key"])

In [0]:
spark.sql("SHOW CATALOGS").show(truncate=False)

In [0]:
import boto3

access_key = dbutils.secrets.get(scope="aws", key="access_key_id")
secret_key = dbutils.secrets.get(scope="aws", key="secret_access_key")

s3 = boto3.client(
    "s3",
    aws_access_key_id=access_key,
    aws_secret_access_key=secret_key,
    region_name="us-east-2",
)

bucket = "eduardo-personal-data-projects"
prefix = "raw/nyc_taxi/"

response = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)
keys = [obj["Key"] for obj in response.get("Contents", []) if obj["Key"].endswith(".parquet")]
print(f"{len(keys)} arquivos encontrados:", keys)

In [0]:
import pandas as pd
from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "bronze"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

for key in keys:
    local_path = f"/tmp/{key.split('/')[-1]}"
    s3.download_file(bucket, key, local_path)

    pdf = pd.read_parquet(local_path)

    df = (
        spark.createDataFrame(pdf)
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_file", F.lit(key))
    )

    (
        df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(f"{CATALOG}.{SCHEMA}.nyc_taxi_trips")
    )

    print(f"Gravado: {key}")

print("Ingestão bronze concluída.")

In [0]:
spark.sql("SELECT COUNT(*) AS total_linhas, MIN(_ingested_at) AS primeiro_ingest, COUNT(DISTINCT _source_file) AS arquivos_distintos FROM workspace.bronze.nyc_taxi_trips").show()

In [0]:
spark.sql("select * from workspace.bronze.nyc_taxi_trips").show()

In [0]:
spark.sql("select * from workspace.bronze.nyc_taxi_trips").describe()

In [0]:
df = spark.table("workspace.bronze.nyc_taxi_trips")

In [0]:
df.printSchema()      # mostra as colunas e tipos, tipo um df.info() do pandas
df.count()             # total de linhas
display(df.limit(20))  # visualização bonita, com paginação (melhor que .show())

In [0]:
df.describe().show()

In [0]:
total = df.count()
df.describe().filter("summary = 'count'").show(truncate=False)

In [0]:
from pyspark.sql import functions as F

df = spark.table("workspace.bronze.nyc_taxi_trips")

nulos = df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns
])

display(nulos)